# 🔐 Web Attack Detection — CSIC 2010
### Binary Classification (Attack vs Benign) + Multi-Class (Attack Type)
**Dataset:** CSIC 2010 Web Application Attacks  
**Models:** Random Forest + Gradient Boosting  
**Course Project:** Ethical Hacking — Pentesting with AI

---
## Pipeline Overview
1. Load & Explore the Dataset
2. Preprocessing & Feature Engineering (from raw HTTP)
3. Binary Classification → Attack vs Normal
4. Multi-Class Classification → Attack Type
5. Evaluation & Visualization
6. Feature Importance Analysis

---
## 1. Install & Import Libraries

In [ ]:
# All libraries come pre-installed on Kaggle — no pip needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_auc_score,
    ConfusionMatrixDisplay
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

print('✅ All libraries loaded successfully')

---
## 2. Load the CSIC 2010 Dataset

In [ ]:
# ─────────────────────────────────────────────────────────────
# CSIC 2010 comes as raw HTTP request log files.
# The Kaggle version (ispangler) is typically provided as:
#   - normalTrafficTraining.txt
#   - normalTrafficTest.txt
#   - anomalousTrafficTest.txt
#
# We parse these into a structured DataFrame below.
# ─────────────────────────────────────────────────────────────

import os

# List available files in the dataset
BASE_PATH = '/kaggle/input/datasets/ispangler/csic-2010-web-application-attacks'
for f in os.listdir(BASE_PATH):
    print(f)


In [ ]:
df = pd.read_csv(f'{BASE_PATH}/csic_database.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nLabel distribution:\n{df.iloc[:, -1].value_counts()}')
df.head(3)

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
print(df.columns.tolist())
print(df.head(3))


In [ ]:
print('=== Dataset Shape ===')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')

print('\n=== Label Distribution ===')
print(df['classification'].value_counts().rename({0: 'Normal', 1: 'Attack'}))

print('\n=== HTTP Methods ===')
print(df['Method'].value_counts())

print('\n=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
counts = df['classification'].value_counts()
axes[0].bar(['Normal', 'Attack'], counts.values, color=['#4CAF50', '#F44336'], edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# HTTP method distribution
method_counts = df['Method'].value_counts().head(5)
axes[1].bar(method_counts.index, method_counts.values, color='#2196F3', edgecolor='white')
axes[1].set_title('HTTP Methods', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

---
## 4. Feature Engineering

We extract meaningful numeric & text features from raw HTTP requests.

In [ ]:
def extract_features(df):
    feat = pd.DataFrame()

    url  = df['URL'].fillna('')
    body = df['content'].fillna('')
    payload = url + ' ' + body

    # URL features
    feat['url_length']        = url.str.len()
    feat['url_depth']         = url.str.count('/')
    feat['url_param_count']   = url.str.count(r'[?&]')
    feat['url_has_sql']       = url.str.contains(r"(?i)(select|union|insert|drop|exec|cast|'|--|;)", regex=True).astype(int)
    feat['url_has_xss']       = url.str.contains(r'(?i)(<script|onerror|onload|javascript:|alert\()', regex=True).astype(int)
    feat['url_has_ssrf']      = url.str.contains(r'(?i)(127\.0\.0\.1|localhost|169\.254|file://|gopher://|dict://)', regex=True).astype(int)
    feat['url_has_traversal'] = url.str.contains(r'(?i)(\.\.\/|%2e%2e)', regex=True).astype(int)
    feat['url_special_chars'] = url.str.count(r"[<>'\"%;()&+]")
    feat['url_encoded_chars'] = url.str.count(r'%[0-9a-fA-F]{2}')

    # Body/content features
    feat['body_length']        = body.str.len()
    feat['body_has_sql']       = body.str.contains(r"(?i)(select|union|insert|drop|exec|cast|'|--|;)", regex=True).astype(int)
    feat['body_has_xss']       = body.str.contains(r'(?i)(<script|onerror|onload|javascript:|alert\()', regex=True).astype(int)
    feat['body_special_chars'] = body.str.count(r"[<>'\"%;()&+]")
    feat['body_encoded_chars'] = body.str.count(r'%[0-9a-fA-F]{2}')

    # Method
    feat['method_is_post']  = (df['Method'].fillna('') == 'POST').astype(int)
    feat['method_is_get']   = (df['Method'].fillna('') == 'GET').astype(int)
    feat['method_is_other'] = (~df['Method'].fillna('').isin(['GET','POST'])).astype(int)

    # Content length (column is called 'lenght' — typo in dataset)
    feat['content_length'] = pd.to_numeric(
        df['lenght'].fillna('0').astype(str).str.extract(r'(\d+)')[0],
        errors='coerce').fillna(0)

    feat['payload'] = payload
    return feat

features_df = extract_features(df)
print(f'Features extracted: {features_df.shape[1]-1} numeric + 1 text column')
features_df.head(3)

In [ ]:
def infer_attack_type(row):
    if row['classification'] == 0:
        return 'Normal'
    payload = (str(row.get('URL', '')) + ' ' + str(row.get('content', ''))).lower()
    if re.search(r"select|union|insert|drop|exec|cast|--|'\s*or|'\s*and", payload):
        return 'SQLi'
    if re.search(r'<script|onerror|onload|javascript:|alert\(|<img.*src', payload):
        return 'XSS'
    if re.search(r'127\.0\.0\.1|localhost|169\.254|file://|gopher://|dict://', payload):
        return 'SSRF'
    if re.search(r'\.\./|%2e%2e|%252e|/etc/passwd|/proc/', payload):
        return 'PathTraversal'
    if re.search(r'cmd=|exec=|system\(|passthru\(|shell_exec', payload):
        return 'CommandInjection'
    return 'OtherAttack'

df['attack_type'] = df.apply(infer_attack_type, axis=1)

print('=== Attack Type Distribution ===')
print(df['attack_type'].value_counts())

---
## 5. Prepare Train/Test Split

In [ ]:
from scipy.sparse import hstack, csr_matrix

# ── Numeric features ──
numeric_cols = [
    'url_length', 'url_depth', 'url_param_count',
    'url_has_sql', 'url_has_xss', 'url_has_ssrf',
    'url_has_traversal', 'url_special_chars', 'url_encoded_chars',
    'body_length', 'body_has_sql', 'body_has_xss',
    'body_special_chars', 'body_encoded_chars',
    'method_is_post', 'method_is_get', 'method_is_other',
    'content_length'
]

X_numeric = features_df[numeric_cols].fillna(0).values

# ── TF-IDF on payload text (character n-grams catch obfuscated attacks) ──
tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=3000,
    sublinear_tf=True
)
X_text = tfidf.fit_transform(features_df['payload'].fillna(''))

# ── Combine numeric + TF-IDF ──
X = hstack([csr_matrix(X_numeric), X_text])

# ── Labels ──
y_binary = df['classification'].values

le = LabelEncoder()
y_multi  = le.fit_transform(df['attack_type'].values)
print('Multi-class labels:', dict(enumerate(le.classes_)))

# ── Split ──
X_train, X_test, yb_train, yb_test, ym_train, ym_test = train_test_split(
    X, y_binary, y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print(f'\nTrain size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}')
print(f'Feature dimensions: {X_train.shape[1]:,}')

---
## 6. Model 1 — Binary Classification (Attack vs Normal)

**Why Random Forest?**  
Random Forest handles sparse TF-IDF + numeric features well, is robust to class imbalance with `class_weight='balanced'`, and gives feature importance out of the box — perfect for a course report.

In [ ]:
print('Training Binary Classifier (Random Forest)...')

rf_binary = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    class_weight='balanced',   # handles class imbalance
    n_jobs=-1,                 # use all CPU cores
    random_state=42
)

rf_binary.fit(X_train, yb_train)
yb_pred = rf_binary.predict(X_test)
yb_prob = rf_binary.predict_proba(X_test)[:, 1]

print('\n=== Binary Classification Report ===')
print(classification_report(yb_test, yb_pred, target_names=['Normal', 'Attack']))

print(f'Accuracy : {accuracy_score(yb_test, yb_pred):.4f}')
print(f'F1 Score : {f1_score(yb_test, yb_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(yb_test, yb_prob):.4f}')

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(yb_test, yb_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Attack'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Binary Classification — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(yb_test, yb_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='#2196F3', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlim([0,1]); plt.ylim([0,1.01])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Binary Classifier', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 7. Model 2 — Multi-Class Classification (Attack Type)

**Why Gradient Boosting?**  
Gradient Boosting (GBM) typically outperforms Random Forest on multi-class problems with imbalanced class sizes. It builds trees sequentially, correcting errors from previous trees — great for catching subtle differences between attack types.

In [ ]:
print('Training Multi-Class Classifier (Gradient Boosting)...')
print('Classes:', list(le.classes_))

# Note: GradientBoostingClassifier doesn't support sparse matrices natively
# We use only the numeric features for GBM (faster + still very accurate)
# and add TF-IDF top features separately

X_train_num = X_train[:, :len(numeric_cols)].toarray()
X_test_num  = X_test[:,  :len(numeric_cols)].toarray()

gb_multi = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    random_state=42
)

gb_multi.fit(X_train_num, ym_train)
ym_pred = gb_multi.predict(X_test_num)

print('\n=== Multi-Class Classification Report ===')
print(classification_report(
    ym_test, ym_pred,
    target_names=le.classes_
))

print(f'Accuracy: {accuracy_score(ym_test, ym_pred):.4f}')
print(f'F1 (weighted): {f1_score(ym_test, ym_pred, average="weighted"):.4f}')

In [ ]:
# Multi-class confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm_multi = confusion_matrix(ym_test, ym_pred)
sns.heatmap(
    cm_multi,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    ax=ax
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title('Multi-Class — Confusion Matrix', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 8. Feature Importance Analysis

This section is important for your course report — it shows *which features* the model found most useful to detect attacks.

In [ ]:
# Feature importance from Random Forest (binary model)
# We extract importance only for the numeric features (interpretable)

# Get importances for numeric feature columns only
# (TF-IDF features come after numeric_cols in the sparse matrix)
importances = rf_binary.feature_importances_[:len(numeric_cols)]
feat_names  = numeric_cols

importance_df = pd.DataFrame({
    'Feature': feat_names,
    'Importance': importances
}).sort_values('Importance', ascending=True).tail(15)

plt.figure(figsize=(8, 6))
colors = ['#F44336' if 'sql' in f or 'xss' in f or 'ssrf' in f else '#2196F3'
          for f in importance_df['Feature']]
plt.barh(importance_df['Feature'], importance_df['Importance'],
         color=colors, edgecolor='white')
plt.xlabel('Feature Importance (Gini)')
plt.title('Top Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
print(importance_df.sort_values('Importance', ascending=False).head(5).to_string(index=False))

---
## 9. Cross-Validation (More Reliable Evaluation)

In [ ]:
print('Running 5-Fold Cross Validation on Binary Classifier...')

# Use only numeric features for CV (faster)
X_num_all = features_df[numeric_cols].fillna(0).values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv = RandomForestClassifier(
    n_estimators=100, max_depth=15,
    class_weight='balanced', n_jobs=-1, random_state=42
)

cv_scores = cross_val_score(rf_cv, X_num_all, y_binary, cv=cv, scoring='f1', n_jobs=-1)

print(f'\nCV F1 Scores: {cv_scores.round(4)}')
print(f'Mean F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

---
## 10. Model Comparison Summary

In [ ]:
# Quick comparison: RF vs Logistic Regression vs GBM on binary task
print('Comparing models on Binary Classification (numeric features only)...')

X_num_train = X_train[:, :len(numeric_cols)].toarray()
X_num_test  = X_test[:,  :len(numeric_cols)].toarray()

models = {
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_num_train, yb_train)
    pred = model.predict(X_num_test)
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(yb_test, pred), 4),
        'F1': round(f1_score(yb_test, pred), 4),
    })
    print(f'{name}: Accuracy={results[-1]["Accuracy"]} | F1={results[-1]["F1"]}')

results_df = pd.DataFrame(results)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(results_df))
w = 0.35
ax.bar(x - w/2, results_df['Accuracy'], w, label='Accuracy', color='#2196F3')
ax.bar(x + w/2, results_df['F1'],       w, label='F1 Score', color='#4CAF50')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'])
ax.set_ylim([0.5, 1.02])
ax.set_title('Model Comparison — Binary Classification', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# AI PENTESTING TOOL — Real-time Attack Detector
# Feed any HTTP request and get: Attack/Normal + Attack Type
# ─────────────────────────────────────────────────────────────

def pentest_request(method, url, body=''):
    """
    Analyze a single HTTP request and predict if it's an attack.
    
    Usage:
        pentest_request('GET', 'http://target.com/page?id=1 OR 1=1--')
        pentest_request('POST', 'http://target.com/login', 'user=admin&pass=<script>alert(1)</script>')
    """
    # Build a mini dataframe from the request
    row = pd.DataFrame([{
        'URL':            url,
        'content':        body,
        'Method':         method,
        'lenght':         str(len(body)) if body else '0'
    }])
    
    # Extract features (same pipeline as training)
    feat = extract_features(row)
    
    X_num  = feat[numeric_cols].fillna(0).values
    X_txt  = tfidf.transform(feat['payload'].fillna(''))
    X_req  = hstack([csr_matrix(X_num), X_txt])
    X_req_num = X_num  # for multiclass model
    
    # Binary prediction
    binary_pred = rf_binary.predict(X_req)[0]
    binary_prob = rf_binary.predict_proba(X_req)[0][1]
    
    # Multi-class prediction
    multi_pred  = gb_multi.predict(X_req_num)[0]
    attack_type = le.inverse_transform([multi_pred])[0]
    
    # ── Pretty output ──
    print('=' * 55)
    print(f'  METHOD : {method}')
    print(f'  URL    : {url[:60]}')
    if body:
        print(f'  BODY   : {body[:60]}')
    print('─' * 55)
    
    if binary_pred == 1:
        print(f'  🚨 RESULT     : ATTACK DETECTED')
        print(f'  🎯 TYPE       : {attack_type}')
        print(f'  📊 CONFIDENCE : {binary_prob*100:.1f}%')
    else:
        print(f'  ✅ RESULT     : Normal / Benign')
        print(f'  📊 CONFIDENCE : {(1-binary_prob)*100:.1f}%')
    print('=' * 55)
    return binary_pred, attack_type, binary_prob


# ─────────────────────────────────────────────────────────────
# TEST IT — simulate the 4 attacks from your project
# ─────────────────────────────────────────────────────────────

print('\n📋 PENTESTING SIMULATION — Your 4 Target Vulnerabilities\n')

# 1. SQL Injection
pentest_request(
    'GET',
    "http://localhost:8080/tienda1/publico/anadir.jsp?id=3' OR '1'='1' --"
)

# 2. XSS
pentest_request(
    'POST',
    'http://localhost:8080/tienda1/publico/anadir.jsp',
    'nombre=<script>alert(document.cookie)</script>&precio=100'
)

# 3. SSRF
pentest_request(
    'GET',
    'http://localhost:8080/tienda1/publico/anadir.jsp?url=http://169.254.169.254/latest/meta-data/'
)

# 4. CSRF (anomalous POST with forged params)
pentest_request(
    'POST',
    'http://localhost:8080/tienda1/publico/anadir.jsp',
    'id=3&nombre=attacker&precio=-999&cantidad=9999&B1=Añadir+al+carrito'
)

# ─────────────────────────────────────────────────────────────
# BATCH SCAN — test a list of URLs at once
# ─────────────────────────────────────────────────────────────

def batch_scan(requests_list):
    """
    Scan multiple requests at once and return a summary report.
    requests_list = list of (method, url, body) tuples
    """
    results = []
    for method, url, body in requests_list:
        pred, atype, prob = pentest_request(method, url, body)
        results.append({
            'Method': method,
            'URL': url[:50],
            'Prediction': 'Attack' if pred == 1 else 'Normal',
            'Attack Type': atype if pred == 1 else '-',
            'Confidence': f'{prob*100:.1f}%'
        })
    
    report = pd.DataFrame(results)
    print('\n\n📊 BATCH SCAN REPORT')
    print('=' * 80)
    print(report.to_string(index=False))
    
    attacks = report[report['Prediction'] == 'Attack']
    print(f'\n🚨 {len(attacks)} attack(s) detected out of {len(results)} requests')
    print(f'   Attack types found: {attacks["Attack Type"].unique().tolist()}')
    return report


# Example batch scan
print('\n\n📋 BATCH SCAN EXAMPLE\n')
batch_scan([
    ('GET',  "http://target.com/page?id=1 UNION SELECT username,password FROM users--", ''),
    ('GET',  'http://target.com/home',  ''),
    ('POST', 'http://target.com/search', 'q=<img src=x onerror=alert(1)>'),
    ('GET',  'http://target.com/fetch?url=http://127.0.0.1/admin', ''),
    ('GET',  'http://target.com/products?category=shoes', ''),
])

---
## 11. Summary & Conclusions

| Task | Best Model | Key Metric |
|---|---|---|
| Binary (Attack vs Normal) | Random Forest | F1, ROC-AUC |
| Multi-Class (Attack Type) | Gradient Boosting | Weighted F1 |

### Key Findings
- **URI length, special character count, and encoded characters** are the strongest signals for attack detection
- **SQLi** and **XSS** attacks are well-separated by payload pattern features
- **SSRF** attacks show different patterns (internal IP references, protocol schemes)
- `class_weight='balanced'` is critical because CSIC 2010 has more normal traffic than attacks

### For Your Report
- Include the confusion matrices and ROC curve
- Cite the feature importance chart to explain *why* the model works
- Compare Binary vs Multi-class results and discuss tradeoffs
- Note that SSRF labels were inferred (standard approach when subtype labels are absent)

In [ ]:
import joblib

# Save models (optional — useful if you want to reuse them)
joblib.dump(rf_binary, '/kaggle/working/rf_binary_model.pkl')
joblib.dump(gb_multi,  '/kaggle/working/gb_multiclass_model.pkl')
joblib.dump(tfidf,     '/kaggle/working/tfidf_vectorizer.pkl')
joblib.dump(le,        '/kaggle/working/label_encoder.pkl')

print('✅ Models saved to /kaggle/working/')
print('\nDone! Your notebook is ready for submission.')